In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
%cd /workspace/Code/experimental/

/workspace/Code/experimental


In [20]:
from data.datamodule import TextDataModule
from src.modules.jepa.encoder import TransformerEncoder
from src.modules.jepa.mlm_module import MLMModule
import lightning as L
import torch
from data.tokenizer import TokenizerWrapper
from lightning.pytorch.loggers import WandbLogger

In [31]:
PATH = './logs/mlm_baseline/f6ld3szr/checkpoints/epoch=9-step=11030.ckpt'

In [32]:
# 1. Tu instancies d'abord ton encodeur (vierge)
encoder = TransformerEncoder(
    vocab_size=30522,
    d_model=32,
    n_heads=2,
    n_layers=2,
    d_ff=256,
    max_seq_len=128,
)


dm = TokenizerWrapper()
# 2. Tu injectes cet encodeur lors du chargement du checkpoint
model = MLMModule.load_from_checkpoint(
    PATH, 
    encoder=encoder  # <-- On fournit l'argument manquant ici
)

# 3. Mode évaluation
model.eval()
model.freeze()

Chargement du tokenizer depuis le cache local : ./data/tokenizers/bert-base-uncased


MLMModule(
  (encoder): TransformerEncoder(
    (token_embedding): Embedding(30522, 32, padding_idx=0)
    (position_embedding): Embedding(128, 32)
    (embed_dropout): Dropout(p=0.1, inplace=False)
    (blocks): ModuleList(
      (0-1): 2 x TransformerBlock(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (ff): Sequential(
          (0): Linear(in_features=32, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=32, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (final_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  )
  (head): MLMHead(
    (norm): LayerNorm((32,), eps=1e-05, elem

In [37]:
# 0. Récupération du tokenizer
tokenizer = dm.tokenizer  

text = "The capital [MASK] France is [MASK], but I prefer [MASK] [MASK] [MASK]."

# 1. Tokenisation
inputs = tokenizer.encode(text)  

# 2. Construction de l'attention mask
attention_mask_list = [1 if x != tokenizer.pad_token_id else 0 for x in inputs]

print("inputs")
print(inputs)
print(attention_mask_list)

# 3. Trouver TOUS les indices des tokens [MASK]
mask_token_id = tokenizer.mask_token_id

# On liste toutes les positions où apparaît le token [MASK]
mask_indices = [i for i, token_id in enumerate(inputs) if token_id == mask_token_id]

if not mask_indices:
    raise ValueError(f"Le token de masquage ({tokenizer.mask_token}) est introuvable.")

# 4. Conversion en tenseurs avec dimension Batch (B=1)
input_ids = torch.tensor([inputs], dtype=torch.long, device=model.device)                  # Shape: (1, L)
attention_mask = torch.tensor([attention_mask_list], dtype=torch.long, device=model.device)  # Shape: (1, L)

# masked_positions devient de taille (1, M) où M est le nombre de masques trouvés
masked_positions = torch.tensor([mask_indices], dtype=torch.long, device=model.device)      # Shape: (1, M)

print("batch inputs id", input_ids)
print("attn mask", attention_mask)
print("masked pos", masked_positions)

# 5. Passage dans le modèle
model.eval()
with torch.no_grad():
    hidden_states = model.encoder(input_ids, attention_mask)
    
    # Ta fonction extrait les M positions valides -> gathered a une shape (M, D) car N = M ici (Batch=1)
    gathered, valid = model._gather_masked_positions(hidden_states, masked_positions, attention_mask)
    
    # La tête MLM projette vers le vocabulaire -> logits a une shape (M, vocab_size)
    logits = model.head(gathered) 
    
    # Argmax sur la dimension du vocabulaire (-1) pour obtenir les IDs des M tokens prédits
    predicted_token_ids = torch.argmax(logits, dim=-1).tolist()  # Converti en liste Python pour le décodage

# 6. Décodage et affichage du résultat
# On décode chaque token individuellement
predicted_words = [tokenizer.decode([token_id]).strip() for token_id in predicted_token_ids]

print(f"Indices des masques détectés : {mask_indices}")
print(f"Mots prédits dans l'ordre    : {predicted_words}")

# Reconstruction visuelle de la phrase
text_sections = text.split(tokenizer.mask_token)
completed_text = ""
for i, section in enumerate(text_sections[:-1]):
    completed_text += f"{section}**{predicted_words[i]}**"
completed_text += text_sections[-1]

print("\nPhrase complétée :")
print(completed_text)

inputs
[101, 1996, 3007, 103, 2605, 2003, 103, 1010, 2021, 1045, 9544, 103, 103, 103, 1012, 102]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
batch inputs id tensor([[ 101, 1996, 3007,  103, 2605, 2003,  103, 1010, 2021, 1045, 9544,  103,
          103,  103, 1012,  102]])
attn mask tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
masked pos tensor([[ 3,  6, 11, 12, 13]])
Indices des masques détectés : [3, 6, 11, 12, 13]
Mots prédits dans l'ordre    : ['the', 'the', 'the', 'the', 'the']

Phrase complétée :
The capital **the** France is **the**, but I prefer **the** **the** **the**.


In [ ]:
import torch

def predict_with_datamodule(text: str, model: L.LightningModule, dm: TokenizerWrapper) -> list[str]:
    model.eval()
    model.freeze()
    
    tokenizer = dm
    
    # 1. On applique le découpage/formatage brut du dataset de base
    # On simule un batch brut avec un seul texte
    raw_item = dm.train_dataset.masker.apply(dm.train_dataset.tokenizer.encode(text))
    
    # 2. On passe par le collate_fn du DataModule pour appliquer le left-padding exact !
    batch = dm.collate_fn([raw_item])
    
    # On envoie les tenseurs sur le bon device (CPU/GPU)
    input_ids = batch["input_ids"].to(model.device)
    attention_mask = batch["attention_mask"].to(model.device)
    masked_positions = batch["masked_positions"].to(model.device)

    # 3. Inférence
    with torch.no_grad():
        hidden_states = model.encoder(input_ids, attention_mask)
        gathered, valid = model._gather_masked_positions(hidden_states, masked_positions, attention_mask)
        logits = model.head(gathered)
        predicted_token_ids = torch.argmax(logits, dim=-1).tolist()

    # 4. Décodage
    predicted_words = [tokenizer.decode([t]).strip() for t in predicted_token_ids]
    return predicted_words

# --- Test ---
text = "The capital [MASK] France is [MASK]."
words = predict_with_datamodule(text, model, dm)
print("Mots prédits :", words)

AttributeError: 'TokenizerWrapper' object has no attribute 'train_dataset'